# 05 — Wikipedia-Web-Scraping

## Zweck
Wikipedia-Stadtseiten als **Web-Scraping-Quelle** nutzen. Wir laden die rohe HTML-Seite (Bronze),
parsen aus der Infobox **Bevölkerung, Fläche und Bevölkerungsdichte** und speichern sie als Silver.

## Robustheit
Wikipedia-Infoboxen sind uneinheitlich. Fehlende oder mehrdeutige Werte werden **nicht geschätzt**,
sondern bleiben leer. Die Spalte `parse_status` (`success` / `partial` / `failed`) macht transparent,
wie viel je Stadt extrahiert werden konnte.

## Ausgabe
- Bronze: `data/bronze/wikipedia_html/<city_id>.html`
- Silver: `data/silver/city_metadata.parquet`

## Konfiguration

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
HTML_DIR = DATA_DIR / "bronze" / "wikipedia_html"
SILVER_DIR = DATA_DIR / "silver"
HTML_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "euro-air-quality-pipeline/1.0 (didaktisches Scraping-Projekt)"}
city_reference_df = pd.read_parquet(SILVER_DIR / "city_reference.parquet")
print(f"{len(city_reference_df)} Städte aus der Stadtreferenz geladen.")

8 Städte aus der Stadtreferenz geladen.


## HTML laden und in Bronze speichern

In [2]:
for _, row in city_reference_df.iterrows():
    target = HTML_DIR / f"{row['city_id']}.html"
    if target.exists():
        continue   # Bronze-HTML bereits vorhanden -> kein erneuter Wikipedia-Abruf (reproduzierbar, offline-fähig)
    resp = requests.get(row["wikipedia_url"], headers=HEADERS, timeout=20)
    resp.raise_for_status()
    target.write_text(resp.text, encoding="utf-8")
    time.sleep(0.5)   # höflich gegenüber Wikipedia
print(f"{len(city_reference_df)} HTML-Seiten in {HTML_DIR} vorhanden.")

8 HTML-Seiten in /workspace/data/bronze/wikipedia_html vorhanden.


## Parser
Wikipedia-Infoboxen sind aufgebaut aus Abschnitts-Headern (z. B. „Population") mit Unterzeilen
(z. B. „• City 1,982,000"). Deshalb zwei Hilfsfunktionen:

- `clean_number` macht aus `"1,982,000 [1]"` die Zahl `1982000`.
- `section_value` sucht im Abschnitt `section` (z. B. „population") die erste Unterzeile, deren
  Beschriftung zu `labels` passt (z. B. „City", „Total" oder „Density").

In [3]:
def clean_number(text):
    if not text:
        return None
    text = re.sub(r"\[.*?\]", "", text)            # Fußnoten [1] entfernen
    text = text.replace(",", "").replace("\xa0", " ")
    match = re.search(r"[0-9]+(?:\.[0-9]+)?", text)
    if not match:
        return None
    value = float(match.group(0))
    return int(value) if value.is_integer() else value


def normalize(text):
    text = re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip().lower()
    return re.sub(r"^[^a-z0-9]+", "", text)   # führende Aufzählungspunkte/Symbole entfernen


def section_value(soup, section, labels):
    """Im Infobox-Abschnitt 'section' die erste Unterzeile, die mit einer der 'labels' beginnt."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    rows = infobox.select("tr")
    start = None
    for i, tr in enumerate(rows):
        header = tr.select_one("th.infobox-header")
        if header and normalize(header.get_text(" ", strip=True)).startswith(section):
            start = i + 1
            break
    if start is None:
        return None
    texts = []
    for tr in rows[start:]:
        if tr.select_one("th.infobox-header"):   # nächster Abschnitt -> Stop
            break
        texts.append(tr.get_text(" ", strip=True))
    for label in labels:
        for text in texts:
            if normalize(text).startswith(label):
                return text
    return None


def direct_value(soup, label):
    """Kompakte Infoboxen (ohne Abschnitte): Zeile, deren Beschriftung mit 'label' beginnt."""
    infobox = soup.select_one("table.infobox")
    if infobox is None:
        return None
    for tr in infobox.select("tr"):
        th, td = tr.find("th"), tr.find("td")
        if th and td and normalize(th.get_text(" ", strip=True)).startswith(label):
            return td.get_text(" ", strip=True)
    return None


# Stadt-Ebene bevorzugen (nicht Metropolregion).
CITY_LABELS = ["city", "capital", "total", "municipality", "land"]

# Dokumentierte Vergleichbarkeits-Ausnahmen: Wikipedia führt manche Städte als Kernkommune,
# andere als großes Verwaltungsgebiet — Fläche und Dichte sind dann nicht direkt vergleichbar
# (Modifiable Areal Unit Problem). Wir schätzen und korrigieren nichts, sondern markieren die
# Einschränkung transparent über ein Flag, das die Analyse (Notebook 09) auswerten kann.
DENSITY_NOT_COMPARABLE = {
    "paris_fr": "Kernkommune (20 Arrondissements, ~105 km²) — nicht mit den größeren "
                "Verwaltungsflächen der anderen Städte vergleichbar (MAUP).",
}


def parse_city_metadata(row, html):
    soup = BeautifulSoup(html, "html.parser")
    population = clean_number(section_value(soup, "population", CITY_LABELS) or direct_value(soup, "population"))
    area_km2 = clean_number(section_value(soup, "area", CITY_LABELS) or direct_value(soup, "area"))
    density = clean_number(section_value(soup, "population", ["density"]) or direct_value(soup, "density"))
    if density is None and population and area_km2:
        density = population / area_km2

    values = [population, area_km2, density]
    if all(v is not None for v in values):
        status = "success"
    elif any(v is not None for v in values):
        status = "partial"
    else:
        status = "failed"

    return {
        "city_id": row["city_id"],
        "city_name": row["city_name"],
        "country_code": row["country_code"],
        "population": population,
        "area_km2": area_km2,
        "population_density": density,
        "density_comparable": row["city_id"] not in DENSITY_NOT_COMPARABLE,
        "area_basis_note": DENSITY_NOT_COMPARABLE.get(row["city_id"], ""),
        "metadata_source": "wikipedia",
        "parse_status": status,
        "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    }

## Alle Städte parsen

In [4]:
records = [parse_city_metadata(row, (HTML_DIR / f"{row['city_id']}.html").read_text(encoding="utf-8"))
           for _, row in city_reference_df.iterrows()]
city_metadata_df = pd.DataFrame(records)
city_metadata_df[["city_id", "population", "area_km2", "population_density", "density_comparable", "parse_status"]]

,city_id,population,area_km2,population_density,density_comparable,parse_status
0,vienna_at,2028499,414.78,4890.50,True,success
1,berlin_de,3596999,891.30,4109.00,True,success
2,paris_fr,2047602,105.40,19430.00,False,success
3,madrid_es,3477497,605.77,5740.60,True,success
4,rome_it,2746984,1287.36,2133.81,True,success
5,amsterdam_nl,933680,219.32,5277.00,True,success
6,warsaw_pl,1862402,517.24,3500.00,True,success
7,prague_cz,1407084,496.21,2835.70,True,success


## Validierung und Speichern
Plausibilitätsgrenzen: vorhandene Werte müssen positiv sein. Jede Stadt muss genau einen Eintrag haben.

In [5]:
assert city_metadata_df["city_id"].is_unique, "Doppelte city_id in den Metadaten."
assert set(city_metadata_df["city_id"]) == set(city_reference_df["city_id"]), "Stadtmenge weicht ab."
for col in ["population", "area_km2", "population_density"]:
    assert city_metadata_df[col].dropna().gt(0).all(), f"Nicht-positive Werte in {col}."

output_path = SILVER_DIR / "city_metadata.parquet"
city_metadata_df.to_parquet(output_path, index=False)
print(f"Geschrieben: {output_path.name} | parse_status: {city_metadata_df['parse_status'].value_counts().to_dict()}")

Geschrieben: city_metadata.parquet | parse_status: {'success': 8}


## Nächster Schritt
Notebook `06` ausführen — Open-Meteo-Live-Werte abrufen und als Kafka-Producer senden.